# P2PNet + MovingDroneCrowd++ Colab workflow
Run this notebook from top to bottom for setup and smoke tests. Full training/evaluation cells are guarded by explicit switches. Code is cloned to `/content`, data is staged to `/content`, and persistent outputs go to Google Drive.

In [ ]:
# ---------- CENTRAL CONFIGURATION: EDIT THIS CELL ----------
from pathlib import Path

REPO_URL = "https://github.com/YOUR_ACCOUNT/YOUR_P2PNET_FORK.git"
REPO_REF = "your-mdc-branch"  # branch, tag, or commit
REPO_DIR = Path("/content/P2PNet_MDC")

DRIVE_PROJECT = Path("/content/drive/MyDrive/P2PNet_MDC")
DATA_SOURCE = "archive"  # archive | drive_folder | already_staged
DATASET_ARCHIVE = DRIVE_PROJECT / "data/MovingDroneCrowd++.tar"
DRIVE_DATASET_FOLDER = DRIVE_PROJECT / "data/MovingDroneCrowd++"
DATASET_ROOT = Path("/content/data/MovingDroneCrowd++")

RUN_NAME = "p2pnet_mdc_run_001"
RUN_DIR = DRIVE_PROJECT / "runs" / RUN_NAME
TRAIN_SPLIT = "train.txt"
VAL_SPLIT = "val.txt"
TEST_SPLIT = "test.txt"

# Practical starting values; tune on validation, not test.
EPOCHS = 100
LR_DROP = 80
BATCH_SIZE = 1
CROP_SIZE = 512
NUM_PATCHES = 1
TRAIN_FRAME_STRIDE = 1
PERIODIC_VAL_FRAME_STRIDE = 5
EVAL_FREQ = 5
SAVE_EVERY = 10
THRESHOLD = 0.5
TILE_SIZE = 1024
TILE_OVERLAP = 128
NUM_WORKERS = 2

RUN_FULL_TRAINING = False
RUN_RESUME_TRAINING = False
RUN_FINAL_EVALUATION = False
RUN_BATCH_INFERENCE = False

## 1. Mount Drive and inspect the runtime

In [ ]:
from google.colab import drive
import shutil, subprocess, sys, os, json, shlex, platform
drive.mount("/content/drive")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=False)
usage = shutil.disk_usage("/content")
print(f"/content free: {usage.free / 2**30:.1f} GiB")

## 2. Clone the modified repository and install lightweight dependencies
The Colab runtime cannot see ordinary files on your Windows disk. Push this modified repository first, then configure `REPO_URL` and `REPO_REF`.

In [ ]:
if "YOUR_ACCOUNT" in REPO_URL or REPO_REF == "your-mdc-branch":
    raise ValueError("Set REPO_URL and REPO_REF in the configuration cell first.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Reusing existing runtime checkout:", REPO_DIR)
subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)

def run(command):
    command = [str(value) for value in command]
    print("$", shlex.join(command))
    return subprocess.run(command, cwd=REPO_DIR, check=True)

import torch, torchvision
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__, "CUDA:", torch.cuda.is_available())

## 3. Stage MovingDroneCrowd++ on `/content`
`archive` is recommended. `drive_folder` works but performs many Drive file operations. `already_staged` only verifies an existing runtime path.

In [ ]:
import tarfile
DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
if DATASET_ROOT.exists():
    print("Dataset already staged:", DATASET_ROOT)
elif DATA_SOURCE == "archive":
    if not DATASET_ARCHIVE.is_file():
        raise FileNotFoundError(DATASET_ARCHIVE)
    archive_size = DATASET_ARCHIVE.stat().st_size
    free = shutil.disk_usage("/content").free
    if free < archive_size * 2.2:
        raise RuntimeError(f"Not enough /content disk: free={free/2**30:.1f} GiB, archive={archive_size/2**30:.1f} GiB")
    local_archive = Path("/content") / DATASET_ARCHIVE.name
    print("Copying archive to runtime disk...")
    shutil.copy2(DATASET_ARCHIVE, local_archive)
    print("Extracting...")
    with tarfile.open(local_archive) as archive:
        try:
            archive.extractall(DATASET_ROOT.parent, filter="data")
        except TypeError:
            archive.extractall(DATASET_ROOT.parent)
    local_archive.unlink()
elif DATA_SOURCE == "drive_folder":
    if not DRIVE_DATASET_FOLDER.is_dir():
        raise FileNotFoundError(DRIVE_DATASET_FOLDER)
    print("Copying dataset folder; this can be slow on Drive...")
    shutil.copytree(DRIVE_DATASET_FOLDER, DATASET_ROOT)
elif DATA_SOURCE != "already_staged":
    raise ValueError(f"Unknown DATA_SOURCE={DATA_SOURCE!r}")
if not (DATASET_ROOT / "frames").is_dir() or not (DATASET_ROOT / "annotations").is_dir():
    raise RuntimeError(f"Staged dataset layout is wrong: {DATASET_ROOT}")
print("Dataset ready:", DATASET_ROOT)

## 4. Validate all split mappings and inspect one sample

In [ ]:
run([sys.executable, "validate_mdc.py", "--data_root", DATASET_ROOT,
     "--output", RUN_DIR / "dataset_validation.json", "--images_to_open", "10"])

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))
from crowd_datasets.MDC.mdc import MDCFrameDataset
import matplotlib.pyplot as plt
sample_set = MDCFrameDataset(str(DATASET_ROOT), VAL_SPLIT, train=False, max_samples=1)
image, points, record = sample_set.get_raw_sample(0)
plt.figure(figsize=(14, 8))
plt.imshow(image)
if len(points): plt.scatter(points[:, 0], points[:, 1], s=8, facecolors="none", edgecolors="lime")
plt.title(f"{record.sample_id} — {len(points)} heads")
plt.axis("off"); plt.show()

## 5. Cheap model/data forward-backward smoke test

In [ ]:
OFFICIAL_WEIGHTS = REPO_DIR / "weights/SHTechA.pth"
run([sys.executable, "smoke_test.py", "--data_root", DATASET_ROOT,
     "--split_file", TRAIN_SPLIT, "--weights", OFFICIAL_WEIGHTS,
     "--device", "cuda", "--crop_size", "128"])

## 6. Verified single-image inference path
This uses the included ShanghaiTech checkpoint only as a functionality check.

In [ ]:
SMOKE_INFERENCE_DIR = RUN_DIR / "smoke_inference"
run([sys.executable, "run_test.py", "--input", REPO_DIR / "vis/demo1.jpg",
     "--weight_path", OFFICIAL_WEIGHTS, "--output_dir", SMOKE_INFERENCE_DIR,
     "--device", "cuda", "--tile_size", "0", "--max_size", "512"])
from IPython.display import display
from PIL import Image
display(Image.open(next((SMOKE_INFERENCE_DIR / "visualizations").glob("*.jpg"))))

## 7. Tiny two-frame MDC evaluation smoke test

In [ ]:
run([sys.executable, "evaluate.py", "--data_root", DATASET_ROOT,
     "--split_file", VAL_SPLIT, "--weight_path", OFFICIAL_WEIGHTS,
     "--output_dir", RUN_DIR / "smoke_evaluation", "--device", "cuda",
     "--max_samples", "2", "--tile_size", "0", "--max_size", "512",
     "--visualize_first", "1"])

## 8. One-batch training smoke test
This writes only to disposable `/content`; it verifies optimizer/backprop/checkpoint creation.

In [ ]:
run([sys.executable, "train.py", "--dataset_file", "MDC", "--data_root", DATASET_ROOT,
     "--train_split", TRAIN_SPLIT, "--val_split", VAL_SPLIT,
     "--weights", OFFICIAL_WEIGHTS, "--output_dir", "/content/p2pnet_training_smoke",
     "--device", "cuda", "--epochs", "1", "--batch_size", "1",
     "--crop_size", "128", "--num_patches", "1",
     "--max_train_samples", "2", "--max_eval_samples", "1",
     "--max_train_batches", "1", "--eval_freq", "0",
     "--num_workers", "0", "--disable_tensorboard", "--save_every", "0"])
assert Path("/content/p2pnet_training_smoke/checkpoints/latest.pth").is_file()
print("Tiny training checkpoint created successfully.")

## 9. Full training/fine-tuning
Set `RUN_FULL_TRAINING=True` in the configuration cell only after all smoke tests pass. Periodic validation uses every fifth validation frame; final evaluation below uses every frame.

In [ ]:
if not RUN_FULL_TRAINING:
    print("Skipped. Set RUN_FULL_TRAINING=True to start.")
else:
    run([sys.executable, "train.py", "--dataset_file", "MDC", "--data_root", DATASET_ROOT,
         "--train_split", TRAIN_SPLIT, "--val_split", VAL_SPLIT,
         "--weights", OFFICIAL_WEIGHTS, "--output_dir", RUN_DIR,
         "--device", "cuda", "--epochs", EPOCHS, "--lr_drop", LR_DROP,
         "--batch_size", BATCH_SIZE, "--crop_size", CROP_SIZE,
         "--num_patches", NUM_PATCHES, "--min_crop_points", "1",
         "--train_frame_stride", TRAIN_FRAME_STRIDE,
         "--val_frame_stride", PERIODIC_VAL_FRAME_STRIDE,
         "--eval_freq", EVAL_FREQ, "--save_every", SAVE_EVERY,
         "--threshold", THRESHOLD, "--tile_size", TILE_SIZE,
         "--tile_overlap", TILE_OVERLAP, "--num_workers", NUM_WORKERS])

## 10. Resume a persistent run after reset
Set `RUN_RESUME_TRAINING=True`. `EPOCHS` is the final epoch count, not the number of additional epochs.

In [ ]:
LATEST = RUN_DIR / "checkpoints/latest.pth"
if not RUN_RESUME_TRAINING:
    print("Skipped. Set RUN_RESUME_TRAINING=True to resume.")
else:
    if not LATEST.is_file(): raise FileNotFoundError(LATEST)
    run([sys.executable, "train.py", "--dataset_file", "MDC", "--data_root", DATASET_ROOT,
         "--train_split", TRAIN_SPLIT, "--val_split", VAL_SPLIT,
         "--resume", LATEST, "--output_dir", RUN_DIR,
         "--device", "cuda", "--epochs", EPOCHS, "--lr_drop", LR_DROP,
         "--batch_size", BATCH_SIZE, "--crop_size", CROP_SIZE,
         "--num_patches", NUM_PATCHES, "--min_crop_points", "1",
         "--train_frame_stride", TRAIN_FRAME_STRIDE,
         "--val_frame_stride", PERIODIC_VAL_FRAME_STRIDE,
         "--eval_freq", EVAL_FREQ, "--save_every", SAVE_EVERY,
         "--threshold", THRESHOLD, "--tile_size", TILE_SIZE,
         "--tile_overlap", TILE_OVERLAP, "--num_workers", NUM_WORKERS])

## 11. Final full-resolution validation or test
Set `RUN_FINAL_EVALUATION=True`. Change `FINAL_SPLIT` from validation to test only after configuration/threshold choices are frozen.

In [ ]:
FINAL_SPLIT = VAL_SPLIT  # change to TEST_SPLIT for held-out testing
FINAL_CHECKPOINT = RUN_DIR / "checkpoints/best_mae.pth"
if not RUN_FINAL_EVALUATION:
    print("Skipped. Set RUN_FINAL_EVALUATION=True to evaluate every frame.")
else:
    if not FINAL_CHECKPOINT.is_file(): raise FileNotFoundError(FINAL_CHECKPOINT)
    final_name = "validation_full" if FINAL_SPLIT == VAL_SPLIT else "test_full"
    run([sys.executable, "evaluate.py", "--data_root", DATASET_ROOT,
         "--split_file", FINAL_SPLIT, "--weight_path", FINAL_CHECKPOINT,
         "--output_dir", RUN_DIR / final_name, "--device", "cuda",
         "--threshold", THRESHOLD, "--tile_size", TILE_SIZE,
         "--tile_overlap", TILE_OVERLAP, "--frame_stride", "1",
         "--visualize_first", "10"])

## 12. Batch/directory inference

In [ ]:
INFERENCE_INPUT = DATASET_ROOT / "frames/scene_1/1"
INFERENCE_CHECKPOINT = RUN_DIR / "checkpoints/best_mae.pth"
if not RUN_BATCH_INFERENCE:
    print("Skipped. Set RUN_BATCH_INFERENCE=True and choose INFERENCE_INPUT.")
else:
    run([sys.executable, "run_test.py", "--input", INFERENCE_INPUT, "--recursive",
         "--weight_path", INFERENCE_CHECKPOINT,
         "--output_dir", RUN_DIR / "batch_inference", "--device", "cuda",
         "--threshold", THRESHOLD, "--tile_size", TILE_SIZE,
         "--tile_overlap", TILE_OVERLAP])

## 13. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN_DIR / 'tensorboard'}